In [1]:
!pip install -qU langchain "langchain[openai]"
!pip install -qU langgraph


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\pierr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import uuid
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import MessagesState,StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
llm = init_chat_model("gpt-4.1-mini", temperature=0.5)

In [10]:
def propmt_llm(state:MessagesState):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

graph_builder  = StateGraph(MessagesState)
graph_builder.add_node(propmt_llm)
graph_builder.add_edge(START,'propmt_llm')
graph_builder.add_edge('propmt_llm',END)

chekcpointer = InMemorySaver()

graph = graph_builder.compile(checkpointer=chekcpointer)

config = {'configurable' : {'thread_id':uuid.uuid4()}}

user_message = "Bonjour je m'apelle Pierre "
response = graph.invoke({
    'messages':[
        {"role":"user","content":user_message}
    ]
},config=config)
print(response['messages'][-1].content)

user_message2 = "Qui suis-je ?"
print(graph.invoke({'messages':[{"role":"user","content":user_message2}]},config=config))

Bonjour Pierre ! Comment puis-je vous aider aujourd'hui ?
{'messages': [HumanMessage(content="Bonjour je m'apelle Pierre ", additional_kwargs={}, response_metadata={}, id='a75ec118-52cc-478b-b195-d3b9bb08f34e'), AIMessage(content="Bonjour Pierre ! Comment puis-je vous aider aujourd'hui ?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 14, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_f683aae7de', 'id': 'chatcmpl-EC8J8FzFpZSqhZuXrXmNsskpUY66g', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff74f-0663-7121-9055-aba18f4db166-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'in

#### Agent tools

In [2]:
from typing import Annotated,Sequence,TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage,ToolMessage,HumanMessage,SystemMessage,AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import add_messages
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

In [ ]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],add_messages]
    table_questions : list[str]

@tool
def add(a:int,b:int):
    """Cette fonction ajoute 2 nombres"""
    print("tool executé")
    return a+b

tools = [add]
llm = ChatOpenAI(model ="gpt-4.1-mini").bind_tools(tools)

def reformulateur(state:AgentState)->AgentState:
    system_prompt = SystemMessage(content = "tu es un agent IA qui reponds propose 2 reformulaions de la question questions posées pour faire une recherche vectorielle par la suite ")
    response = llm.invoke([system_prompt]+state["messages"])
    state['table_questions'] = response.content.split("\n")
    print("reformulateur exeuté") 
    print(state['table_questions'])
    return {'messages':[response]}
    

def model_call(state:AgentState)->AgentState:
    system_prompt = SystemMessage(content = "tu es un agent IA qui reponds au mieux aux questions posées")
    state["messages"].append(HumanMessage(content="que fait 24+2"))
    response = llm.invoke([system_prompt]+state["messages"])
    return {'messages':[response]}

def shouldContinue(state:AgentState):
    messages = state["messages"][-1]
    if not messages.tool_calls:
        return "end"
    else : 
        return "continue"


graph = StateGraph(AgentState)
graph.add_node("our_agent",model_call)
graph.add_node("reformulateur",reformulateur)

tool_node = ToolNode(tools = tools)
graph.add_node("tools",tool_node)

graph.add_edge(START,"reformulateur")
graph.add_edge("reformulateur","our_agent")
# graph.add_edge(START,"our_agent")
graph.add_conditional_edges(
    "our_agent",
    shouldContinue,
    {
        "continue":"tools",
        "end":END
    }
)

graph.add_edge("tools","our_agent")
app = graph.compile()
user_message2 = "Que fait 24+1"
print(app.invoke({'messages':[{"role":"user","content":user_message2}]}))

reformulateur exeuté
['1. Quelle est la somme de 24 et 1 ?', "2. Quel est le résultat de l'addition 24 plus 1 ?"]
tool executé
tool executé


AttributeError: 'dict' object has no attribute 'content'